In [1]:
import os 
import numpy as np 
import torch 
import PIL 
import torch.nn as nn 
import torch.nn.functional  as F 
from torch.utils.data import Dataset,DataLoader,dataloader, random_split
from PIL import Image
from tqdm import tqdm

In [2]:
TRAIN_DATA_DIR = '/home/aum-thaker/Desktop/VSC/recordai/data/train_images'
# TEST_DATA_DIR = '/home/aum-thaker/Desktop/VSC/recordai/data/test_images'

## using a simple cnn siameese network

In [3]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),   # make all images same size
    transforms.ToTensor()
])

In [4]:
class RecaiBase(Dataset):

    def __init__(self, data_dir_path, transform=None):

        self.authentic_dir = os.path.join(data_dir_path, "authentic")
        self.forged_dir = os.path.join(data_dir_path, "forged")

        self.authentic_images = os.listdir(self.authentic_dir)
        self.forged_images = os.listdir(self.forged_dir)

        self.transform = transform

    def __len__(self):
        return min(len(self.authentic_images), len(self.forged_images))

    def __getitem__(self, idx):

        auth_path = os.path.join(self.authentic_dir, self.authentic_images[idx])
        forged_path = os.path.join(self.forged_dir, self.forged_images[idx])

        image_authentic = Image.open(auth_path).convert("RGB")
        image_forged = Image.open(forged_path).convert("RGB")

        if self.transform:
            image_authentic = self.transform(image_authentic)
            image_forged = self.transform(image_forged)

        return image_authentic, image_forged

In [5]:
dataset = RecaiBase(TRAIN_DATA_DIR,transform)

In [6]:
dataset_size = len(dataset)

train_size = int(0.8 * dataset_size)
test_size = dataset_size - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [8]:
class SiameseCNN(nn.Module):

    def __init__(self):
        super(SiameseCNN, self).__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(128*28*28,256),
            nn.ReLU(),

            nn.Linear(256,128)
        )

    def forward_once(self, x):
        return self.encoder(x)

    def forward(self, img1, img2):

        out1 = self.forward_once(img1)
        out2 = self.forward_once(img2)

        return out1, out2

In [9]:
class ContrastiveLoss(nn.Module):

    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, out1, out2, label):

        distance = F.pairwise_distance(out1, out2)

        loss = torch.mean(
            (1-label)*torch.pow(distance,2) +
            label*torch.pow(torch.clamp(self.margin-distance, min=0),2)
        )

        return loss

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [11]:
model = SiameseCNN().to(device)

In [12]:
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
for epoch in range(10):

    model.train()
    total_loss = 0

    loop = tqdm(train_loader)

    for authentic, forged in loop:

        authentic = authentic.to(device)
        forged = forged.to(device)

        label = torch.ones(authentic.size(0), dtype=torch.float32).to(device)

        out1, out2 = model(authentic, forged)

        loss = criterion(out1, out2, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

Epoch 1:  49%|████▊     | 58/119 [00:14<00:14,  4.20it/s, loss=0.0424]  

In [ ]:
model.eval()

distances = []

with torch.no_grad():

    for authentic, forged in test_loader:

        authentic = authentic.to(device)
        forged = forged.to(device)

        out1, out2 = model(authentic, forged)

        distance = F.pairwise_distance(out1, out2)

        distances.extend(distance.cpu().numpy())

print("Mean distance:", np.mean(distances))